# LAB 4: Compare Performance of Uninformed vs. Informed Search

**Relates to:** Weeks 4–5 — Evaluating Search Strategies, Heuristics, A\*

## WHAT THIS LAB IS ABOUT

BFS is blind: it treats every direction as equally promising and spreads out evenly in all directions. A* is informed: it gets a
hint, called a heuristic, estimating how far each cell is from the goal, and uses that hint to explore towards the goal first. The
formula A* uses is `f = g + h`, where `g` is the real cost of getting to a cell so far, and `h` is the estimated cost remaining. In this
lab you count how many cells each algorithm examines, proving with numbers that the hint saves work.

## BEFORE YOU START, YOU NEED

• Completion of Lab 3 (you will reuse the maze and the neighbors function)
• Understanding of Python's `heapq` module (explained below)


## Lab 3 code used in this lab

The next cells supply the maze and the two helper functions reused by BFS and A\*. Each helper has its own explanation and code cell.


In [ ]:
# Maze used by the search algorithms

maze = [
    [0, 0, 0, 1, 0],
    [1, 1, 0, 1, 0],
    [0, 0, 0, 0, 0],
    [0, 1, 1, 1, 0],
    [0, 0, 0, 0, 0]
]

start = (0, 0)
goal = (len(maze) - 1, len(maze[0]) - 1)

### Helper 1 — Find walkable neighbors

This function checks the four cells directly above, below, left, and right, returning only cells that are inside the maze and not walls.


In [ ]:
def neighbors(cell, maze):

    """Return walkable cells directly next to cell."""

    row, col = cell

    candidates = [
        (row - 1, col),
        (row + 1, col),
        (row, col - 1),
        (row, col + 1)
    ]

    return [
        (candidate_row, candidate_col)
        for candidate_row, candidate_col in candidates
        if 0 <= candidate_row < len(maze)
        and 0 <= candidate_col < len(maze[0])
        and maze[candidate_row][candidate_col] == 0

    ]

### Helper 2 — Rebuild the final path

During a search, `parent` records where each cell was reached from. This function follows those links backward from the goal, then reverses the result into start-to-goal order.


In [14]:
def reconstruct_path(parent, start, goal):

    """Rebuild a path by following parent links from goal to start."""

    path = [goal]

    while path[-1] != start:

        path.append(parent[path[-1]])

    path.reverse()

    return path

## Building it step by step

### Step 1 — Write the heuristic

Manhattan distance counts how many steps you would need if there were no walls at all: the horizontal gap plus the vertical
gap. It is called admissible because it never overestimates — walls can only make the real journey longer, never shorter.
Admissibility is what guarantees A\* still finds the shortest path.


In [15]:
def manhattan(a, b):
    return abs(a[0] - b[0]) + abs(a[1] - b[1])

# Example:
print(manhattan((0,0), (4,4))) # 4 + 4 = 8
print(manhattan((2,3), (4,4))) # 2 + 1 = 3

8
3


**Line by line:**

- `abs(...)` Absolute value — turns a negative difference positive
- `a[0] - b[0]` The vertical (row) gap
- `a[1] - b[1]` The horizontal (column) gap
- `Never overestimates` Because ignoring walls can only make the estimate too SMALL, never too big


### Step 2 — Add counters to BFS

To compare fairly we need a number. We add a variable called `expanded` that increases by one every time we take a cell off
the frontier and examine it. This is the standard measure of search effort.


In [16]:
from collections import deque

def bfs_counted(maze, start, goal):
    frontier = deque([start])
    visited = {start}
    parent = {}
    expanded = 0 # <-- the counter
    
    while frontier:
        current = frontier.popleft()
        expanded += 1 # <-- count this examination
        
        if current == goal:
            path = reconstruct_path(parent, start, goal)
            return path, expanded
            
        for nxt in neighbors(current, maze):
            if nxt not in visited:
                visited.add(nxt)
                parent[nxt] = current
                frontier.append(nxt)
                
    return None, expanded

**Line by line:**

- `expanded = 0` Starts the tally at zero
- `expanded += 1` Shorthand for `expanded = expanded + 1`
- `return path, expanded` Returns two things at once, as a tuple


### Step 3 — Understand the priority queue

A\* must always examine the most promising cell next, not the oldest or newest. A priority queue does exactly that: no matter
what order you put things in, `heappop` always hands back the item with the smallest first value. Try this snippet on its own to
see it work.


In [17]:
import heapq

queue = []
heapq.heappush(queue, (5, 'expensive'))
heapq.heappush(queue, (1, 'cheap'))
heapq.heappush(queue, (3, 'medium'))

print(heapq.heappop(queue)) # (1, 'cheap') <- smallest comes out first
print(heapq.heappop(queue)) # (3, 'medium')
print(heapq.heappop(queue)) # (5, 'expensive')

(1, 'cheap')
(3, 'medium')
(5, 'expensive')


**Line by line:**

- `heapq.heappush(queue, item)` Adds an item, keeping the queue ordered
- `heapq.heappop(queue)` Removes and returns the SMALLEST item
- `(5, 'expensive')` A tuple; heapq sorts by the first element


### Step 4 — Write A\* search

Each frontier entry is `(f, g, cell)`. We always pop the lowest `f`. `g_cost` remembers the cheapest known cost to reach each cell,
so if we find a better route to a cell we already saw, we update it. That check is what keeps A\* correct.


In [ ]:
def a_star(maze, start, goal):

    # Each entry is (f_score, g_score, cell)
    frontier = [(manhattan(start, goal), 0, start)]
    g_cost = {start: 0}
    parent = {}
    expanded = 0

    while frontier:
        
        f, g, current = heapq.heappop(frontier)
        expanded += 1

        if current == goal:
            path = reconstruct_path(parent, start, goal)
            return path, expanded

        for nxt in neighbors(current, maze):
            new_g = g + 1 # every step costs 1

            # Only continue if this is the cheapest route found so far

            if nxt not in g_cost or new_g < g_cost[nxt]:
                g_cost[nxt] = new_g
                f_score = new_g + manhattan(nxt, goal)
                heapq.heappush(frontier, (f_score, new_g, nxt))
                parent[nxt] = current

    return None, expanded

**Line by line:**

- `(manhattan(start,goal), 0, start)` At the start: g is 0, so f equals h
- `f, g, current = heapq.heappop(...)` Unpacks the three values from the lowest-f entry
- `new_g = g + 1` The real cost to reach the neighbour: one more step
- `f_score = new_g + manhattan(nxt, goal)` THE A\* FORMULA: f = g + h
- `new_g < g_cost[nxt]` We found a cheaper route to a cell we already knew about


### Step 5 — Time both and print a comparison table

`time.perf_counter()` gives a high-precision timestamp. Subtracting two of them gives the elapsed seconds; we multiply by
1000 to show milliseconds.


In [ ]:
import time

t0 = time.perf_counter()

bfs_path, bfs_expanded = bfs_counted(maze, start, goal)

t1 = time.perf_counter()


t2 = time.perf_counter()

astar_path, astar_expanded = a_star(maze, start, goal)

t3 = time.perf_counter()



print(f"{'Algorithm':<12}{'PathLen':<10}{'Expanded':<12}{'Time(ms)':<10}")

print('-' * 44)

print(f"{'BFS':<12}{len(bfs_path):<10}{bfs_expanded:<12}{(t1-t0)*1000:<10.3f}")

print(f"{'A*':<12}{len(astar_path):<10}{astar_expanded:<12}{(t3-t2)*1000:<10.3f}")

saving = 100 * (1 - astar_expanded / bfs_expanded)

print()

print(f'A* examined {saving:.1f}% fewer cells than BFS.')

print('Both found a path of the same length, so A* lost no quality.')

Algorithm   PathLen   Expanded    Time(ms)  
--------------------------------------------
BFS         9         15          0.110     
A*          9         9           0.122     

A* examined 40.0% fewer cells than BFS.
Both found a path of the same length, so A* lost no quality.


## WHAT YOU SHOULD SEE

```
Algorithm   PathLen   Expanded    Time(ms)
--------------------------------------------
BFS         9         16          0.021
A*          9         11          0.018

A* examined 31.2% fewer cells than BFS.
Both found a path of the same length, so A* lost no quality.
```

(Exact timings vary by machine. The key result is the Expanded column:
A\* should always be lower than or equal to BFS, while PathLen stays
identical — proving the heuristic saved work without costing quality.)

## IF SOMETHING GOES WRONG

| Error message / symptom                                                 | Why it happened                                                                  | How to fix it                                                                   |
| :---------------------------------------------------------------------- | :------------------------------------------------------------------------------- | :------------------------------------------------------------------------------ |
| `TypeError: '<' not supported between instances of 'tuple' and 'tuple'` | Two frontier entries had identical f and g, so Python tried to compare the cells | Add a counter as a third tie-breaker value, as shown in Lab 5                   |
| A\* returns a longer path than BFS                                      | The heuristic overestimates — it is not admissible                               | Make sure you use Manhattan distance, not Euclidean squared or a scaled version |
| Both report the same Expanded count                                     | The maze is too small or too open for the heuristic to help                      | Try a larger maze with a longer, more winding corridor                          |
| ZeroDivisionError computing saving                                      | bfs_expanded was zero, meaning the search never ran (0)                          | Check that start and goal are both on walkable cells                            |

## TRY IT YOURSELF

• Set the heuristic to always return 0 and re-run. A* should now expand exactly as many cells as BFS — because with no hint, A* degenerates into uninformed search.
• Multiply the heuristic by 3. The path may get longer, showing what happens when a heuristic is not admissible.
• Build a 20x20 maze and see how much larger the saving becomes.
• Add Greedy Best-First Search (use only h, ignoring g) and add it as a third row in the table.
